In [1]:
# Env + install
import sys, torch
print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
print("GPU(s):", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

# Cài các gói cần (không cài torch/torchvision để tránh xung đột)
# !pip -q install -U ultralytics opencv-python wandb


Python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
CUDA available: True
GPU(s): 2 ['Tesla T4', 'Tesla T4']


In [9]:
# Set Kaggle API if kaggle.json is attached via "Add data"
import os, glob, json, shutil, pathlib
kc = pathlib.Path("/root/.config/kaggle"); kc.mkdir(parents=True, exist_ok=True)
matches = glob.glob("/kaggle/input/**/kaggle.json", recursive=True)
if matches:
    shutil.copy(matches[0], kc/"kaggle.json"); os.chmod(kc/"kaggle.json", 0o600)
    creds = json.load(open(kc/"kaggle.json"))
    os.environ["KAGGLE_USERNAME"] = creds.get("username",""); os.environ["KAGGLE_KEY"] = creds.get("key","")
    print("Kaggle API configured from:", matches[0])
else:
    print("No kaggle.json found in /kaggle/input — OK if dataset is already added as Input.")


Kaggle API configured from: /kaggle/input/car-detection-kaggle/kaggle.json


In [10]:
# Fetch dataset
import os, pathlib, glob
ROOT = pathlib.Path("/kaggle/working")
os.chdir(ROOT)
DST = ROOT/"data/car_detection"; DST.mkdir(parents=True, exist_ok=True)

# 1) Nếu bạn đã Add dataset "seyeon040768/car-detection-dataset"
candidates = sorted(glob.glob("/kaggle/input/car-detection-dataset*")) + sorted(glob.glob("/kaggle/input/*car*detect*"))
used = None
for d in candidates:
    if os.path.isdir(d) and glob.glob(d+"/*"):
        used = d; break

if used:
    print("Using input dataset:", used)
    !cp -r "$used"/* "data/car_detection/"
elif (ROOT/"car-detection-dataset.zip").exists():
    print("Found local zip → unzip")
    !unzip -o car-detection-dataset.zip -d data/car_detection >/dev/null
else:
    print("Downloading via Kaggle API…")
    !kaggle datasets download -d seyeon040768/car-detection-dataset -p .
    !unzip -o car-detection-dataset.zip -d data/car_detection >/dev/null
    !rm -f car-detection-dataset.zip

# Preview tree
!find data/car_detection -maxdepth 3 -type d | sort | sed -n '1,80p'


Using input dataset: /kaggle/input/car-detection-kaggle
data/car_detection


In [12]:
# Auto-detect YOLO dataset (train/images) từ /kaggle/input; nếu không có thì tải qua Kaggle API
import os, glob, pathlib, yaml, subprocess

def find_yolo_roots(base_dir:str):
    roots=set()
    for pat in ("**/train/images", "**/images/train"):
        for p in pathlib.Path(base_dir).glob(pat):
            roots.add(p.parents[1])  # dataset root
    return sorted(roots, key=lambda p: len(str(p)))  # path ngắn trước

# 1) Tìm trong /kaggle/input (chỉ nhận dataset có cấu trúc YOLO)
candidates = find_yolo_roots("/kaggle/input")

source = None
if candidates:
    DATA_ROOT = str(candidates[0])
    source = "input"
else:
    # 2) Không có trong input → tải qua Kaggle API (cần Internet ON + kaggle.json OK)
    print("No YOLO-structured dataset found in /kaggle/input → downloading via Kaggle API...")
    work = pathlib.Path("/kaggle/working"); os.chdir(work)
    dst = work/"data/car_detection"; dst.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(["kaggle","datasets","download","-d","seyeon040768/car-detection-dataset","-p","/kaggle/working"], check=False)
    if r.returncode != 0:
        raise SystemExit("❌ Không có dataset YOLO trong /kaggle/input và tải API thất bại. Hãy Add data: 'seyeon040768/car-detection-dataset'.")
    subprocess.run(["bash","-lc","unzip -o /kaggle/working/car-detection-dataset.zip -d /kaggle/working/data/car_detection >/dev/null"], check=False)
    pathlib.Path("/kaggle/working/car-detection-dataset.zip").unlink(missing_ok=True)
    candidates = find_yolo_roots("/kaggle/working/data/car_detection")
    assert candidates, "❌ Đã unzip nhưng vẫn không thấy train/images."
    DATA_ROOT = str(candidates[0])
    source = "downloaded"

def has(p): return os.path.isdir(os.path.join(DATA_ROOT, p))
train = "train/images" if has("train/images") else "images/train"
val   = "valid/images" if has("valid/images") else ("val/images" if has("val/images") else None)
assert val, f"❌ Dataset thiếu valid/val trong: {DATA_ROOT}"
test  = "test/images"  if has("test/images")  else val

# Ghi configs/car_detection.yaml
os.makedirs("configs", exist_ok=True)
DATA_YAML = "configs/car_detection.yaml"
with open(DATA_YAML, "w") as f:
    yaml.safe_dump({"path": DATA_ROOT, "train": train, "val": val, "test": test, "nc": 1, "names": ["car"]}, f, sort_keys=False)

print("✅ Wrote", DATA_YAML)
print(open(DATA_YAML).read())
print("Source:", source, "| DATA_ROOT:", DATA_ROOT)


No YOLO-structured dataset found in /kaggle/input → downloading via Kaggle API...
Dataset URL: https://www.kaggle.com/datasets/seyeon040768/car-detection-dataset
License(s): other


100%|██████████| 886M/886M [00:00<00:00, 1.19GB/s]



✅ Wrote configs/car_detection.yaml
path: /kaggle/working/data/car_detection/car_dataset-master
train: train/images
val: valid/images
test: test/images
nc: 1
names:
- car

Source: downloaded | DATA_ROOT: /kaggle/working/data/car_detection/car_dataset-master


In [ ]:
pip install -q "numpy<2.1,>=1.26.4" "matplotlib>=3.7,<3.9" "protobuf>=4.25.1" --upgrade

In [15]:
from ultralytics import YOLO
import torch, os

# Tắt W&B (bật thì đổi 'true' -> 'false' và wandb.login())
os.environ['WANDB_DISABLED'] = 'true'

ngpu = torch.cuda.device_count()
device = "0,1" if ngpu >= 2 else (0 if ngpu == 1 else "cpu")
batch = 16 * (2 if ngpu >= 2 else 1)

print("GPUs:", ngpu, "| device:", device, "| batch:", batch)

model = YOLO("yolov8n.pt")
results = model.train(
    data="configs/car_detection.yaml",
    epochs=30,
    batch=batch,
    imgsz=640,
    device=device,      # "0,1" sẽ tự chạy DDP trên 2 T4
    workers=4,
    project="runs_detect",
    name="car_detection_online",
    optimizer="AdamW",
    lr0=0.001, lrf=0.01, weight_decay=0.0005,
    patience=10, plots=True, verbose=True
)


GPUs: 2 | device: 0,1 | batch: 32
Ultralytics 8.3.186 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
                                                        CUDA:1 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=configs/car_detection.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=car_detection_online2, nbs=64, nms=Fa

In [ ]:
# Nén folder thành zip
!zip -r car_detection_online2.zip /kaggle/working/runs_detect/car_detection_online2

# Sau khi chạy, ở sidebar bên phải (tab "Output" hoặc "Files") bạn sẽ thấy file zip này
# → Bấm chuột phải → Download


Ultralytics 8.3.186 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
